# Stage 3 - DPO Preference Alignment

**Project:** PrepMind - GenAI / Agentic AI Learning Assistant
**Base model:** Stage-2 SFT adapter (`outputs/sft_adapter`) on top of `Qwen2.5-1.5B-Instruct`
**Goal:** Use Direct Preference Optimization (DPO) on the preference dataset
(`data/preference_dataset.jsonl`) to push the model further toward answers that are
correct, helpful, safe, professional, and domain-specific - and away from answers that
are wrong, generic, unsafe, or rude.

Run on a Colab **T4 GPU** runtime.


## 1. Install dependencies

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets


## 2. Clone the project repo

In [ ]:
REPO_URL = "https://github.com/Abhishek15016/prepmind.git"
!git clone $REPO_URL project
%cd project


## 3. Load the Stage-2 SFT model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
SFT_ADAPTER_PATH = "outputs/sft_adapter"  # from notebook 2, upload/mount if needed

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_PATH,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="qwen2.5")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


## 4. Load and format the preference dataset

`data/preference_dataset.jsonl` stores each example as `prompt` (a list of chat
messages), `chosen` (list with the winning assistant turn), and `rejected` (list with the
losing assistant turn). `DPOTrainer` expects `prompt`, `chosen`, and `rejected` as plain
strings rendered through the chat template.

In [ ]:
from datasets import load_dataset

raw_pref = load_dataset("json", data_files="data/preference_dataset.jsonl", split="train")
print(raw_pref)
print(raw_pref[0])


In [ ]:
def format_pref(example):
    prompt_text = tokenizer.apply_chat_template(
        example["prompt"], tokenize=False, add_generation_prompt=True
    )
    chosen_text = example["chosen"][0]["content"]
    rejected_text = example["rejected"][0]["content"]
    return {"prompt": prompt_text, "chosen": chosen_text, "rejected": rejected_text}

pref_ds = raw_pref.map(format_pref, remove_columns=raw_pref.column_names)
pref_ds = pref_ds.train_test_split(test_size=0.1, seed=42)
print(pref_ds["train"][0]["prompt"][:400])
print("CHOSEN:", pref_ds["train"][0]["chosen"][:200])
print("REJECTED:", pref_ds["train"][0]["rejected"][:200])


## 5. Configure and run DPO training

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="outputs/dpo",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=2,
    learning_rate=5e-6,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    seed=42,
    beta=0.1,               # DPO temperature - how strongly to penalize the rejected response
    max_length=1024,
    max_prompt_length=512,
    save_strategy="epoch",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Unsloth reuses the base weights under the LoRA adapter as the implicit reference
    args=dpo_config,
    train_dataset=pref_ds["train"],
    eval_dataset=pref_ds["test"],
    tokenizer=tokenizer,
)

dpo_stats = dpo_trainer.train()


## 6. Save the final DPO-aligned model

In [ ]:
model.save_pretrained("outputs/dpo_adapter")
tokenizer.save_pretrained("outputs/dpo_adapter")
print("Saved DPO LoRA adapter to outputs/dpo_adapter")

# Merge LoRA into the base weights for a single, easy-to-serve model
model.save_pretrained_merged("outputs/dpo_merged", tokenizer, save_method="merged_16bit")
print("Saved merged fp16 model to outputs/dpo_merged")

# Optional: push to Hugging Face Hub for use in src/inference.py and the Gradio app
# model.push_to_hub_merged("your-username/prepmind-dpo-qwen2.5-1.5b", tokenizer, save_method="merged_16bit")


## 7. Test the DPO-aligned model on the same 10 evaluation questions

In [ ]:
EVAL_QUESTIONS = [
    "What is the difference between LoRA and QLoRA?",
    "Why does self-attention need positional encoding?",
    "What is Retrieval-Augmented Generation and why does it reduce hallucination?",
    "Explain the difference between an LLM agent and a simple prompted LLM call.",
    "What is DPO, and how is it different from RLHF with PPO?",
    "Why do we chunk documents before embedding them for RAG?",
    "What is the KV cache and why does it speed up LLM inference?",
    "What is Model Context Protocol (MCP) and what problem does it solve?",
    "How would you evaluate a RAG system end-to-end?",
    "What is catastrophic forgetting, and why is LoRA less prone to it than full fine-tuning?",
]

FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=300):
    messages = [
        {"role": "system", "content": "You are a friendly expert tutor in Generative AI and Agentic AI. Explain concepts in depth with intuitive analogies, practical production insight, and interview-ready takeaways, in a clear and engaging style."},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    output = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True)
    text = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    return text.strip()

for q in EVAL_QUESTIONS:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 80)


## Notes

- `beta` in `DPOConfig` controls how strongly the loss penalizes the rejected response
  relative to the frozen reference policy - lower beta (e.g. 0.1) allows more movement
  away from the SFT model, higher beta keeps the DPO model closer to it.
- Copy the printed Q/A pairs above into `reports/final_evaluation.md` alongside the base
  and SFT answers to build the three-way comparison table.
- `outputs/dpo_merged` is the final artifact `src/inference.py` and the Gradio app
  (`src/app.py`) are designed to load.


## 8. Launch the side-by-side Gradio comparison app

Compares the Base model, the SFT model, and this DPO model on the same question at once,
in three columns. Uses `src/app.py` from the repo you cloned in step 2.

In [ ]:
!pip install -q gradio
!python src/app.py --sft_adapter outputs/sft_adapter --dpo_adapter outputs/dpo_adapter --share
